In [1]:
from pymilvus import  (connections,utility,FieldSchema,CollectionSchema,DataType,Collection,utility)
from langchain_milvus import Milvus
import numpy as np
from model import RagLLM, RagEmbedding
from doc_parse import chunk,read_and_process_excel,logger
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid


connections.connect(host='127.0.0.1', port="19530")
print("链接成功")

链接成功


In [2]:
collections = utility.list_collections()
print(collections)
print("ok")

['rag_db']
ok


In [ ]:
pdf_files = ['/data/raggggg/员工制度rag/data/zhidu_employee.pdf','/data/raggggg/员工制度rag/data/zhidu_travel.pdf']
excel_files = ['/data/raggggg/员工制度rag/data/zhidu_detail.xlsx']
r_spliter = RecursiveCharacterTextSplitter(chunk_size=128,
                                          chunk_overlap=30,
                                          separators=["\n\n",
                                                      "\n", 
                                                      ".",
                                                      "\uff0e",
                                                       "\u3002",
                                                      ",",
                                                      "\uff0c", 
                                                      "\u3001",
                                                     ])

In [ ]:
# import nltk
# nltk.download('wordnet')

In [6]:
doc_data = []
for pdf_file_name in pdf_files:
    res = chunk(pdf_file_name, callback=logger)
    for data in res:
        content = data["content_with_weight"]
        if '<table>' not in content and len(content) > 200:
            doc_data.extend(r_spliter.split_text(content))
        else:
            doc_data.append(content)
print(f"pdf done,{len(doc_data)}条chunk")


OCR is running...


OCR finished.
OCR: 1.1166346850000082
preprocess
Layout analysis finished.
layouts: 1.3284680980000303
preprocess
Table analysis finished.
Text merging finished
OCR is running...


OCR finished.
OCR: 0.5106579849999662
preprocess
Layout analysis finished.
layouts: 0.6556853469999169
preprocess
Table analysis finished.
Text merging finished
pdf done,38条chunk


In [ ]:
for excel_file_name in excel_files:
    data = read_and_process_excel(excel_file_name)
    df = pd.DataFrame(data[8:], columns=data[7])
    data_excel = df.drop(columns=df.columns[11:17])
    content = data_excel.to_markdown(index=False).replace(' ', "")
    doc_data.append(
        f"### 以下是中央和国家机关工作人员赴地方差旅住宿费标准明细表：\n\n{content}"
    )
print(f"总文档块数量: {len(doc_data)}")

总文档块数量: 39


In [8]:
print(doc_data[:2])

['<table><caption>病假发放标准：</caption>\n<tr><td  >病假时间</td><td  >连续工龄</td><td  >发放标准</td></tr>\n<tr><td></td><td  >不满二年</td><td  >60% </td></tr>\n<tr><td></td><td  >已满二年不满四年</td><td  >70% </td></tr>\n<tr><td  >6 个月以内病假</td><td  >已满四年不满六年</td><td  >80% </td></tr>\n<tr><td></td><td  >已满六年不满八年</td><td  >90% </td></tr>\n<tr><td></td><td  >八年以上</td><td  >100% </td></tr>\n<tr><td></td><td  >不满一年</td><td  >40% </td></tr>\n<tr><td  >6 个月以上病假</td><td  >已满一年不满三年</td><td  >50% </td></tr>\n<tr><td></td><td  >连续工龄三年以上</td><td  >60% </td></tr>\n</table>', '教职工考勤管理制度\n第一节适用范围\n1、本制度包括了考勤、休假、加班等方面的规定。2、本制度适用于学校全体教职员工。']


In [9]:
documents = []
for idx, chunk_text in enumerate(doc_data):
    is_table = 1 if "table" in chunk_text.lower() or idx == len(doc_data) - 1 else 0
    
    doc = Document(
        page_content=chunk_text,
        metadata={
            "type": "ori",
            "is_table": is_table,
            "source": "zhidu_rules"
        }
    )
    documents.append(doc)

In [10]:
embedding_cls = RagEmbedding()
embedding_function = embedding_cls.get_embedding_fun()


/data/raggggg/员工制度rag/model.py:58: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  self.embedding = HuggingFaceEmbeddings(model_name=model_path,


In [11]:
documents = [Document(page_content=text) for text in doc_data]

In [12]:
vector_db = Milvus.from_documents(
    documents=documents,
    embedding=embedding_function,
    collection_name='rag_db',
    connection_args={
        "host": '127.0.0.1',
        "port": '19530'
    },
    # 可选：自定义索引参数
    index_params={
        "metric_type": "L2",   # 或 "L2", "IP"
        "index_type": "HNSW",      # 常用索引类型
        "params": {"M": 16, "efConstruction": 200}
    },
)

In [ ]:
print(f"Milvus构建完成！集合名称: {'rag_db'}")
print(f"共存入 {len(documents)} 条向量")

Milvus 向量库构建完成！集合名称: rag_db
共存入 39 条向量


In [15]:
query = '迟到有什么规定'

related_docs = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(related_docs):
    print(f"\n--- 检索结果 {i+1} ---")
    print(doc.page_content[:3] + "..." if len(doc.page_content) > 300 else doc.page_content)
    print("Metadata:", doc.metadata)


--- 检索结果 1 ---
第二节考勤规定
Metadata: {'pk': 465404275887967252}

--- 检索结果 2 ---
教职工考勤管理制度
第一节适用范围
1、本制度包括了考勤、休假、加班等方面的规定。2、本制度适用于学校全体教职员工。
Metadata: {'pk': 465404275887967251}

--- 检索结果 3 ---
第五节违规违纪处理
1. 迟到、早退(1)劳动考勤是公司支付薪资的依据和职工年度岗位考核内容之一。（2）迟到或早退60分钟以上（含60 分钟），每次视同缺勤1 天。（3）职工迟到、早退、脱岗累计超过3 次的（含），从第1 次起，每次扣减工资50 元。
Metadata: {'pk': 465404275887967274}
